# McLaren Race Intelligence Platform — Data Update

This notebook keeps the lab's F1 dataset current. It finds the latest round already
in GCS, fetches any newer rounds from the Jolpica API, and overwrites the affected
Parquet files in GCS. Students loading data in the lab will automatically get the
latest available data.

**Run this notebook before each lab event to freshen the data.**

### Tables updated

| Table | Source | Notes |
|-------|--------|-------|
| `races` | Jolpica | New race entries |
| `results` | Jolpica | Race results |
| `qualifying` | Jolpica | Qualifying results |
| `sprint_results` | Jolpica | Sprint race results |
| `driver_standings` | Jolpica | Round-by-round championship standings |
| `constructor_standings` | Jolpica | Round-by-round constructor standings |
| `constructor_results` | Derived | Per-constructor points per race, rebuilt from results |
| `drivers` | Jolpica | New entrants added automatically |
| `constructors` | Jolpica | New entrants added automatically |
| `circuits` | Jolpica | Full overwrite from circuits endpoint |
| `seasons` | Derived | New season year entries added |
| `mclaren_drivers` | Derived | Career stats rebuilt from updated results |

### Tables NOT updated

| Table | Reason |
|-------|--------|
| `lap_times` | Jolpica does not expose lap-by-lap timing data |
| `pit_stops` | Jolpica does not expose pit stop data |

These two tables retain Kaggle coverage through late 2024. The Ergast API
(which Jolpica mirrors) never included this data in its public endpoints.

**Requirements:** Colab Enterprise (IAM auth automatic), write access to `gs://class-demo`

---
## Section 0: Setup

In [1]:
!pip install -q google-cloud-storage pandas pyarrow requests

In [2]:
import google.auth
import pandas as pd
import numpy as np
import requests
import datetime
import time
import io
from google.cloud import storage

credentials, PROJECT_ID = google.auth.default()

GCS_BUCKET     = "class-demo"
GCS_PREFIX     = "mclaren-f1"
GCS_BQ_STAGING = f"{GCS_PREFIX}/bq-staging"
MCLAREN_REF    = "mclaren"
JOLPICA_BASE   = "https://api.jolpi.ca/ergast/f1"
RATE_SLEEP     = 3.0
RETRY_BACKOFF  = 30.0
MAX_RETRIES    = 3

gcs_client = storage.Client(project=PROJECT_ID)

print(f"✅ Project  : {PROJECT_ID}")
print(f"✅ GCS      : gs://{GCS_BUCKET}/{GCS_BQ_STAGING}/")
print(f"✅ Run date : {datetime.date.today()}")


def fetch_json(url, retries=MAX_RETRIES):
    """Fetch a Jolpica URL with 429-aware backoff."""
    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=30)
            if resp.status_code == 200:
                return resp.json()
            elif resp.status_code == 429:
                wait = RETRY_BACKOFF * (attempt + 1)
                print(f"  429 — waiting {wait:.0f}s...")
                time.sleep(wait)
            else:
                print(f"  HTTP {resp.status_code} for {url}")
                return None
        except Exception as e:
            print(f"  Request error: {e}")
            time.sleep(RATE_SLEEP * 2)
    print(f"  ⚠️  Failed after {retries} attempts: {url}")
    return None

✅ Project  : qwiklabs-gcp-02-b36a0d0ed605
✅ GCS      : gs://class-demo/mclaren-f1/bq-staging/
✅ Run date : 2026-04-02


---
## Section 1: Load Current Parquet Files from GCS

In [3]:
TABLES_TO_LOAD = [
    "races", "results", "qualifying", "sprint_results",
    "driver_standings", "constructor_standings", "constructor_results",
    "drivers", "constructors", "circuits", "seasons", "status",
    "mclaren_drivers",
]

bucket = gcs_client.bucket(GCS_BUCKET)
tables = {}

print("Loading tables from GCS...\n")
for name in TABLES_TO_LOAD:
    blob_path = f"{GCS_BQ_STAGING}/{name}.parquet"
    blob = bucket.blob(blob_path)
    if blob.exists():
        data = blob.download_as_bytes()
        tables[name] = pd.read_parquet(io.BytesIO(data))
        print(f"  ✅ {name:<30} {len(tables[name]):>8,} rows")
    else:
        print(f"  ❌ {name:<30} NOT FOUND")

missing = [t for t in TABLES_TO_LOAD if t not in tables]
if missing:
    raise RuntimeError(f"Cannot continue — missing tables: {missing}")

print(f"\n✅ All {len(tables)} tables loaded")

Loading tables from GCS...

  ✅ races                             1,173 rows
  ✅ results                          27,169 rows
  ✅ qualifying                       10,992 rows
  ✅ sprint_results                      480 rows
  ✅ driver_standings                 35,383 rows
  ✅ constructor_standings            13,642 rows
  ✅ constructor_results              12,625 rows
  ✅ drivers                             865 rows
  ✅ constructors                        214 rows
  ✅ circuits                             78 rows
  ✅ seasons                              75 rows
  ✅ status                              141 rows
  ✅ mclaren_drivers                      55 rows

✅ All 13 tables loaded


---
## Section 2: Find Latest Round and Determine What to Fetch

In [4]:
# Use results as source of truth for latest completed round
results_with_round = tables["results"].merge(
    tables["races"][["race_id", "year", "round"]], on="race_id", how="left"
)
latest_year  = int(results_with_round["year"].max())
latest_round = int(
    results_with_round[results_with_round["year"] == latest_year]["round"].max()
)
print(f"Latest data in GCS : {latest_year} Round {latest_round}")

current_year   = datetime.date.today().year
years_to_check = sorted(set([latest_year, current_year, current_year + 1]))
print(f"Checking years     : {years_to_check}")

rounds_needed = []  # (year, round, race_name)

for year in years_to_check:
    data = fetch_json(f"{JOLPICA_BASE}/{year}?limit=100")
    time.sleep(RATE_SLEEP)
    if not data:
        continue
    try:
        race_list = data["MRData"]["RaceTable"]["Races"]
    except (KeyError, TypeError):
        continue
    for race in race_list:
        rnd = int(race["round"])
        if year < latest_year:
            continue
        if year == latest_year and rnd <= latest_round:
            continue
        race_date = race.get("date", "")
        if race_date:
            try:
                if datetime.date.fromisoformat(race_date) > datetime.date.today():
                    continue
            except ValueError:
                pass
        rounds_needed.append((year, rnd, race.get("raceName", f"Round {rnd}")))

if rounds_needed:
    print(f"\n{len(rounds_needed)} new round(s) to fetch:")
    for y, r, name in rounds_needed:
        print(f"  {y} Round {r:2d} — {name}")
else:
    print("\n✅ Data is already up to date — nothing to fetch.")

Latest data in GCS : 2026 Round 1
Checking years     : [2026, 2027]

2 new round(s) to fetch:
  2026 Round  2 — Chinese Grand Prix
  2026 Round  3 — Japanese Grand Prix


---
## Section 3: Refresh Circuits and Seasons

Circuits is a full overwrite from Jolpica, preserving existing IDs.
Seasons adds any new year entries.

In [5]:
# ── Circuits ───────────────────────────────────────────────────────────────
print("Refreshing circuits...")
all_circuits = []
offset = 0
while True:
    data = fetch_json(f"{JOLPICA_BASE}/circuits?limit=100&offset={offset}")
    time.sleep(RATE_SLEEP)
    if not data:
        break
    batch = data["MRData"]["CircuitTable"]["Circuits"]
    all_circuits.extend(batch)
    total = int(data["MRData"].get("total", len(all_circuits)))
    offset += 100
    if offset >= total:
        break

if all_circuits:
    existing = tables["circuits"]
    ref_col  = "circuit_ref" if "circuit_ref" in existing.columns else "circuit_id"
    ref_to_id = dict(zip(existing[ref_col].astype(str), existing["circuit_id"]))
    next_cid  = int(existing["circuit_id"].max()) + 1

    rows = []
    for c in all_circuits:
        ref = c.get("circuitId", "")
        cid = ref_to_id.get(ref)
        if cid is None:
            cid = next_cid
            next_cid += 1
            print(f"  + New circuit: {c.get('circuitName', ref)}")
        loc = c.get("Location", {})
        rows.append({
            "circuit_id":  cid,
            "circuit_ref": ref,
            "name":        c.get("circuitName", ""),
            "location":    loc.get("locality", ""),
            "country":     loc.get("country", ""),
            "lat":         pd.to_numeric(loc.get("lat"),   errors="coerce"),
            "lng":         pd.to_numeric(loc.get("long"),  errors="coerce"),
            "alt":         pd.to_numeric(loc.get("alt"),   errors="coerce"),
            "url":         c.get("url", ""),
        })
    tables["circuits"] = pd.DataFrame(rows)
    print(f"  ✅ circuits: {len(tables['circuits'])} rows")
else:
    print("  ⚠️  Could not fetch circuits — keeping existing table")


# ── Seasons ────────────────────────────────────────────────────────────────
existing_years = set(tables["seasons"]["year"].astype(int).tolist())
new_season_rows = []
for year in years_to_check:
    if year not in existing_years:
        new_season_rows.append({"year": year, "url": ""})
        print(f"  + New season: {year}")
if new_season_rows:
    tables["seasons"] = pd.concat(
        [tables["seasons"], pd.DataFrame(new_season_rows)], ignore_index=True
    )
print(f"  ✅ seasons: {len(tables['seasons'])} rows")

Refreshing circuits...
  ✅ circuits: 78 rows
  + New season: 2026
  + New season: 2027
  ✅ seasons: 77 rows


---
## Section 4: Fetch New Race Rounds from Jolpica

In [7]:
if not rounds_needed:
    print("✅ Nothing to fetch — skipping.")
else:
    # Lookup dicts
    driver_ref_to_id      = dict(zip(tables["drivers"]["driver_ref"].astype(str),      tables["drivers"]["driver_id"]))
    constructor_ref_to_id = dict(zip(tables["constructors"]["constructor_ref"].astype(str), tables["constructors"]["constructor_id"]))
    circuit_ref_to_id     = dict(zip(tables["circuits"]["circuit_ref"].astype(str),     tables["circuits"]["circuit_id"]))

    sprint_pk = "sprint_result_id" if "sprint_result_id" in tables["sprint_results"].columns else "result_id"

    next_ids = {
        "race_id":                  int(tables["races"]["race_id"].max()) + 1,
        "result_id":                int(tables["results"]["result_id"].max()) + 1,
        "qualify_id":               int(tables["qualifying"]["qualify_id"].max()) + 1,
        "sprint_result_id":         int(tables["sprint_results"][
            "sprint_result_id" if "sprint_result_id" in tables["sprint_results"].columns
            else "result_id"
        ].max()) + 1,
        "driver_standings_id":      int(tables["driver_standings"]["driver_standings_id"].max()) + 1,
        "constructor_standings_id": int(tables["constructor_standings"]["constructor_standings_id"].max()) + 1,
        "driver_id":                int(tables["drivers"]["driver_id"].max()) + 1,
        "constructor_id":           int(tables["constructors"]["constructor_id"].max()) + 1,
    }

    def get_or_create_driver(d):
        ref = d.get("driverId", "")
        if ref in driver_ref_to_id:
            return driver_ref_to_id[ref]
        did = next_ids["driver_id"]
        next_ids["driver_id"] += 1
        tables["drivers"] = pd.concat([tables["drivers"], pd.DataFrame([{
            "driver_id": did, "driver_ref": ref, "number": pd.NA,
            "code": d.get("code",""), "forename": d.get("givenName",""),
            "surname": d.get("familyName",""), "dob": d.get("dateOfBirth",""),
            "nationality": d.get("nationality",""), "url": d.get("url",""),
        }])], ignore_index=True)
        driver_ref_to_id[ref] = did
        print(f"    + New driver: {d.get('givenName','')} {d.get('familyName','')} ({ref})")
        return did

    def get_or_create_constructor(c):
        ref = c.get("constructorId", "")
        if ref in constructor_ref_to_id:
            return constructor_ref_to_id[ref]
        cid = next_ids["constructor_id"]
        next_ids["constructor_id"] += 1
        tables["constructors"] = pd.concat([tables["constructors"], pd.DataFrame([{
            "constructor_id": cid, "constructor_ref": ref,
            "name": c.get("name", ref), "nationality": c.get("nationality",""),
            "url": c.get("url",""),
        }])], ignore_index=True)
        constructor_ref_to_id[ref] = cid
        print(f"    + New constructor: {c.get('name', ref)} ({ref})")
        return cid

    new_races, new_results, new_qualifying = [], [], []
    new_sprint, new_drv_std, new_con_std   = [], [], []

    for year, rnd, race_name in rounds_needed:
        print(f"\n  {year} Round {rnd:2d} — {race_name}")
        race_id = next_ids["race_id"]

        # Race entry
        d = fetch_json(f"{JOLPICA_BASE}/{year}/{rnd}?limit=1")
        time.sleep(RATE_SLEEP)
        if d:
            try:
                r = d["MRData"]["RaceTable"]["Races"][0]
                cref = r["Circuit"]["circuitId"]
                new_races.append({
                    "race_id": race_id, "year": year, "round": rnd,
                    "circuit_id": circuit_ref_to_id.get(cref, pd.NA),
                    "name": r.get("raceName", race_name),
                    "date": r.get("date",""), "time": r.get("time",""), "url": r.get("url",""),
                })
                next_ids["race_id"] += 1
            except (KeyError, IndexError): pass

        # Results
        d = fetch_json(f"{JOLPICA_BASE}/{year}/{rnd}/results?limit=30")
        time.sleep(RATE_SLEEP)
        if d:
            try:
                rl = d["MRData"]["RaceTable"]["Races"]
                if rl:
                    for e in rl[0].get("Results", []):
                        new_results.append({
                            "result_id": next_ids["result_id"], "race_id": race_id,
                            "driver_id": get_or_create_driver(e["Driver"]),
                            "constructor_id": get_or_create_constructor(e["Constructor"]),
                            "number": pd.to_numeric(e.get("number"), errors="coerce"),
                            "grid": int(e.get("grid", 0)),
                            "position": pd.to_numeric(e.get("position"), errors="coerce"),
                            "position_text": e.get("positionText",""),
                            "position_order": int(e.get("positionOrder", 0)),
                            "points": float(e.get("points", 0)),
                            "laps": int(e.get("laps", 0)),
                            "time": e.get("Time",{}).get("time","") if "Time" in e else "",
                            "milliseconds": pd.to_numeric(e.get("Time",{}).get("millis") if "Time" in e else None, errors="coerce"),
                            "fastest_lap": pd.to_numeric(e.get("FastestLap",{}).get("lap") if "FastestLap" in e else None, errors="coerce"),
                            "rank": pd.to_numeric(e.get("FastestLap",{}).get("rank") if "FastestLap" in e else None, errors="coerce"),
                            "fastest_lap_time": e.get("FastestLap",{}).get("Time",{}).get("time","") if "FastestLap" in e else "",
                            "fastest_lap_speed": pd.to_numeric(e.get("FastestLap",{}).get("AverageSpeed",{}).get("speed") if "FastestLap" in e else None, errors="coerce"),
                            "status_id": 1,
                        })
                        next_ids["result_id"] += 1
                    print(f"    results   : {len(rl[0].get('Results',[]))}")
            except (KeyError, IndexError): pass

        # Qualifying
        d = fetch_json(f"{JOLPICA_BASE}/{year}/{rnd}/qualifying?limit=30")
        time.sleep(RATE_SLEEP)
        if d:
            try:
                rl = d["MRData"]["RaceTable"]["Races"]
                if rl:
                    for e in rl[0].get("QualifyingResults", []):
                        new_qualifying.append({
                            "qualify_id": next_ids["qualify_id"], "race_id": race_id,
                            "driver_id": get_or_create_driver(e["Driver"]),
                            "constructor_id": get_or_create_constructor(e["Constructor"]),
                            "number": pd.to_numeric(e.get("number"), errors="coerce"),
                            "position": int(e.get("position", 0)),
                            "q1": e.get("Q1",""), "q2": e.get("Q2",""), "q3": e.get("Q3",""),
                        })
                        next_ids["qualify_id"] += 1
                    print(f"    qualifying : {len(rl[0].get('QualifyingResults',[]))}")
            except (KeyError, IndexError): pass

        # Sprint
        d = fetch_json(f"{JOLPICA_BASE}/{year}/{rnd}/sprint?limit=30")
        time.sleep(RATE_SLEEP)
        if d:
            try:
                rl = d["MRData"]["RaceTable"]["Races"]
                if rl and rl[0].get("SprintResults"):
                    for e in rl[0]["SprintResults"]:
                        new_sprint.append({
                            "sprint_result_id": next_ids["sprint_result_id"], "race_id": race_id,
                            "driver_id": get_or_create_driver(e["Driver"]),
                            "constructor_id": get_or_create_constructor(e["Constructor"]),
                            "number": pd.to_numeric(e.get("number"), errors="coerce"),
                            "grid": int(e.get("grid", 0)),
                            "position": pd.to_numeric(e.get("position"), errors="coerce"),
                            "position_text": e.get("positionText",""),
                            "position_order": int(e.get("positionOrder", 0)),
                            "points": float(e.get("points", 0)),
                            "laps": int(e.get("laps", 0)),
                            "time": e.get("Time",{}).get("time","") if "Time" in e else "",
                            "milliseconds": pd.to_numeric(e.get("Time",{}).get("millis") if "Time" in e else None, errors="coerce"),
                            "fastest_lap_time": e.get("FastestLap",{}).get("Time",{}).get("time","") if "FastestLap" in e else "",
                            "status_id": 1,
                        })
                        next_ids["sprint_result_id"] += 1
                    print(f"    sprint     : {len(rl[0]['SprintResults'])}")
            except (KeyError, IndexError): pass

        # Driver standings
        d = fetch_json(f"{JOLPICA_BASE}/{year}/{rnd}/driverStandings?limit=100")
        time.sleep(RATE_SLEEP)
        if d:
            try:
                sl = d["MRData"]["StandingsTable"]["StandingsLists"]
                if sl:
                    for e in sl[0].get("DriverStandings", []):
                        new_drv_std.append({
                            "driver_standings_id": next_ids["driver_standings_id"],
                            "race_id": race_id,
                            "driver_id": get_or_create_driver(e["Driver"]),
                            "points": float(e.get("points", 0)),
                            "position": int(e.get("position", 0)),
                            "position_text": e.get("positionText",""),
                            "wins": int(e.get("wins", 0)),
                        })
                        next_ids["driver_standings_id"] += 1
                    print(f"    drv_stnd   : {len(sl[0].get('DriverStandings',[]))}")
            except (KeyError, IndexError): pass

        # Constructor standings
        d = fetch_json(f"{JOLPICA_BASE}/{year}/{rnd}/constructorStandings?limit=100")
        time.sleep(RATE_SLEEP)
        if d:
            try:
                sl = d["MRData"]["StandingsTable"]["StandingsLists"]
                if sl:
                    for e in sl[0].get("ConstructorStandings", []):
                        new_con_std.append({
                            "constructor_standings_id": next_ids["constructor_standings_id"],
                            "race_id": race_id,
                            "constructor_id": get_or_create_constructor(e["Constructor"]),
                            "points": float(e.get("points", 0)),
                            "position": int(e.get("position", 0)),
                            "position_text": e.get("positionText",""),
                            "wins": int(e.get("wins", 0)),
                        })
                        next_ids["constructor_standings_id"] += 1
                    print(f"    con_stnd   : {len(sl[0].get('ConstructorStandings',[]))}")
            except (KeyError, IndexError): pass

    # Merge
    print("\n── Merging ───────────────────────────────────────────────────────────")

    def merge_rows(table_name, new_rows, dedup_keys):
        if not new_rows:
            print(f"  {table_name:<30} no new rows")
            return
        combined = pd.concat([tables[table_name], pd.DataFrame(new_rows)], ignore_index=True)
        valid_keys = [k for k in dedup_keys if k in combined.columns]
        if valid_keys:
            combined = combined.drop_duplicates(subset=valid_keys, keep="last")
        tables[table_name] = combined.reset_index(drop=True)
        print(f"  {table_name:<30} +{len(new_rows)} rows → {len(tables[table_name]):,} total")

    merge_rows("races",                new_races,    ["year", "round"])
    merge_rows("results",              new_results,  ["race_id", "driver_id"])
    merge_rows("qualifying",           new_qualifying,["race_id", "driver_id"])
    merge_rows("sprint_results",       new_sprint,   ["race_id", "driver_id"])
    merge_rows("driver_standings",     new_drv_std,  ["race_id", "driver_id"])
    merge_rows("constructor_standings",new_con_std,  ["race_id", "constructor_id"])
    print(f"  {'drivers':<30} {len(tables['drivers']):,} total")
    print(f"  {'constructors':<30} {len(tables['constructors']):,} total")
    print("\n✅ Race data merged")


  2026 Round  2 — Chinese Grand Prix
    results   : 22
    qualifying : 22
    sprint     : 22
    drv_stnd   : 22
    con_stnd   : 11

  2026 Round  3 — Japanese Grand Prix
    results   : 22
    qualifying : 22
    drv_stnd   : 22
    con_stnd   : 11

── Merging ───────────────────────────────────────────────────────────
  races                          +2 rows → 1,173 total
  results                        +44 rows → 27,213 total
  qualifying                     +44 rows → 11,036 total
  sprint_results                 +22 rows → 502 total
  driver_standings               +44 rows → 35,427 total
  constructor_standings          +22 rows → 13,664 total
  drivers                        865 total
  constructors                   214 total

✅ Race data merged


---
## Section 5: Rebuild Derived Tables

Rebuild `constructor_results` and `mclaren_drivers` from the updated results.
Always rebuilt from scratch to stay consistent with the source data.

In [8]:
# ── constructor_results ────────────────────────────────────────────────────
print("Rebuilding constructor_results...")
con_results = (
    tables["results"]
    .groupby(["race_id", "constructor_id"])
    .agg(points=("points", "sum"))
    .reset_index()
)
con_results["constructor_results_id"] = range(1, len(con_results) + 1)
con_results["status"] = "Finished"
tables["constructor_results"] = con_results
print(f"  ✅ constructor_results: {len(tables['constructor_results']):,} rows")


# ── mclaren_drivers ────────────────────────────────────────────────────────
print("\nRebuilding mclaren_drivers...")

mclaren_id = tables["constructors"][
    tables["constructors"]["constructor_ref"] == MCLAREN_REF
].iloc[0]["constructor_id"]

mac_results = tables["results"][tables["results"]["constructor_id"] == mclaren_id].copy()
mac_results = mac_results.merge(
    tables["races"][["race_id", "year", "round", "name", "circuit_id"]], on="race_id"
)

driver_stats = mac_results.groupby("driver_id").agg(
    mclaren_first_year=("year", "min"),
    mclaren_last_year=("year", "max"),
    mclaren_races=("race_id", "count"),
    mclaren_points=("points", "sum"),
).reset_index()

pos_col = "position_order" if "position_order" in mac_results.columns else "position"
pos_numeric = pd.to_numeric(mac_results[pos_col], errors="coerce")
mac_results["_pos"] = pos_numeric

wins_df   = mac_results[mac_results["_pos"] == 1].groupby("driver_id").size().reset_index(name="mclaren_wins")
podium_df = mac_results[mac_results["_pos"] <= 3].groupby("driver_id").size().reset_index(name="mclaren_podiums")

driver_stats = driver_stats.merge(wins_df,   on="driver_id", how="left")
driver_stats = driver_stats.merge(podium_df, on="driver_id", how="left")
driver_stats[["mclaren_wins", "mclaren_podiums"]] = \
    driver_stats[["mclaren_wins", "mclaren_podiums"]].fillna(0).astype(int)

mclaren_drivers = tables["drivers"][
    tables["drivers"]["driver_id"].isin(driver_stats["driver_id"])
].merge(driver_stats, on="driver_id")
mclaren_drivers = mclaren_drivers.sort_values("mclaren_first_year").reset_index(drop=True)

tables["mclaren_drivers"] = mclaren_drivers
print(f"  ✅ mclaren_drivers: {len(tables['mclaren_drivers'])} drivers")

# Spot-check current drivers
print("\n  Current/recent McLaren drivers:")
recent_drivers = mclaren_drivers[
    mclaren_drivers["mclaren_last_year"] >= datetime.date.today().year - 1
].sort_values("mclaren_last_year", ascending=False)
for _, row in recent_drivers.head(6).iterrows():
    print(f"    {row['forename']} {row['surname']:<20} "
          f"{int(row['mclaren_first_year'])}–{int(row['mclaren_last_year'])}  "
          f"{int(row['mclaren_races'])} races  {int(row['mclaren_wins'])} wins")

Rebuilding constructor_results...
  ✅ constructor_results: 13,298 rows

Rebuilding mclaren_drivers...
  ✅ mclaren_drivers: 55 drivers

  Current/recent McLaren drivers:
    Lando Norris               2019–2026  155 races  11 wins
    Oscar Piastri              2023–2026  73 races  9 wins


---
## Section 6: Write Updated Parquet Files to GCS

In [9]:
TABLES_TO_WRITE = [
    "races", "results", "qualifying", "sprint_results",
    "driver_standings", "constructor_standings", "constructor_results",
    "drivers", "constructors", "circuits", "seasons",
    "mclaren_drivers",
]

print(f"Writing {len(TABLES_TO_WRITE)} Parquet files to GCS...\n")
bucket   = gcs_client.bucket(GCS_BUCKET)
total_mb = 0

for name in TABLES_TO_WRITE:
    df  = tables[name]
    buf = io.BytesIO()
    df.to_parquet(buf, index=False)
    size_mb = buf.tell() / 1024 / 1024
    total_mb += size_mb
    buf.seek(0)
    blob_path = f"{GCS_BQ_STAGING}/{name}.parquet"
    bucket.blob(blob_path).upload_from_file(buf, content_type="application/octet-stream")
    print(f"  ✅ {name:<30} {len(df):>8,} rows  ({size_mb:.2f} MB)")

print(f"\n  Total written  : {total_mb:.1f} MB")
print(f"  Not updated    : lap_times, pit_stops (Jolpica API limitation)")
print(f"\n✅ GCS update complete")

Writing 12 Parquet files to GCS...

  ✅ races                             1,173 rows  (0.04 MB)
  ✅ results                          27,213 rows  (0.57 MB)
  ✅ qualifying                       11,036 rows  (0.22 MB)
  ✅ sprint_results                      502 rows  (0.02 MB)
  ✅ driver_standings                 35,427 rows  (0.34 MB)
  ✅ constructor_standings            13,664 rows  (0.13 MB)
  ✅ constructor_results              13,298 rows  (0.10 MB)
  ✅ drivers                             865 rows  (0.05 MB)
  ✅ constructors                        214 rows  (0.01 MB)
  ✅ circuits                             78 rows  (0.01 MB)
  ✅ seasons                              77 rows  (0.00 MB)
  ✅ mclaren_drivers                      55 rows  (0.01 MB)

  Total written  : 1.5 MB
  Not updated    : lap_times, pit_stops (Jolpica API limitation)

✅ GCS update complete


---
## Section 7: Verification

In [10]:
merged = tables["results"].merge(
    tables["races"][["race_id", "year", "round"]], on="race_id", how="left"
)
new_latest_year  = int(merged["year"].max())
new_latest_round = int(merged[merged["year"] == new_latest_year]["round"].max())

print("── Coverage (last 5 seasons) ─────────────────────────────────────────")
recent = merged[merged["year"] >= new_latest_year - 4]
summary = (
    recent.groupby("year")
    .agg(rounds=("round", "nunique"), result_rows=("result_id", "count"))
    .reset_index()
    .sort_values("year", ascending=False)
)
for _, row in summary.iterrows():
    print(f"  {int(row['year'])}: {int(row['rounds']):2d} rounds, {int(row['result_rows']):4d} result rows")

print(f"\n── Final counts ──────────────────────────────────────────────────────")
for name in TABLES_TO_WRITE:
    print(f"  {name:<30} {len(tables[name]):>8,}")
print(f"  {'lap_times (unchanged)':<30} {len(tables.get('lap_times', pd.DataFrame())):>8,}  (Kaggle only)")
print(f"  {'pit_stops (unchanged)':<30} {len(tables.get('pit_stops', pd.DataFrame())):>8,}  (Kaggle only)")

print(f"\n  Latest data : {new_latest_year} Round {new_latest_round}")
print(f"  Run date    : {datetime.date.today()}")
print(f"\n✅ Data update complete")

── Coverage (last 5 seasons) ─────────────────────────────────────────
  2026:  3 rounds,   66 result rows
  2025: 24 rounds,  479 result rows
  2024: 24 rounds,  479 result rows
  2023: 22 rounds,  440 result rows
  2022: 22 rounds,  440 result rows

── Final counts ──────────────────────────────────────────────────────
  races                             1,173
  results                          27,213
  qualifying                       11,036
  sprint_results                      502
  driver_standings                 35,427
  constructor_standings            13,664
  constructor_results              13,298
  drivers                             865
  constructors                        214
  circuits                             78
  seasons                              77
  mclaren_drivers                      55
  lap_times (unchanged)                 0  (Kaggle only)
  pit_stops (unchanged)                 0  (Kaggle only)

  Latest data : 2026 Round 3
  Run date    : 2026-04-02

✅